# Import pandas and veltix_raw_data.csv

In [1]:
import pandas as pd
df = pd.read_csv('data/raw/veltix_raw_data.csv')
print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 8034 rows, 10 columns


# 1. Data Conceptualising
- Data grain: one row = one SKU's weekly sales and inventory snapshot

In [2]:
df

,sku_id,product_category,week_id,sales_qty,inventory_begin,inventory_end,receipts,lead_time_days,unit_cost,holding_cost_rate
0,SH-005,Smart Home,2023-W28,24.0,180.0,156,0,0,69.09,0.25
1,WR_007,Wearables,2024-W12,35.0,466.0,431,0,0,65.79,0.30
2,aU-010,Audio,2024-W17,33.0,1776.0,1743,0,0,57.05,0.28
3,WR-005,Wearables,2023-W52,180.0,492.0,312,0,0,131.41,0.30
4,MA-003,Mobile Accessories,2023-W07,166.0,1659.0,1493,0,0,5.65,0.25
...,...,...,...,...,...,...,...,...,...,...
8029,CP-004,Computer Peripherals,2024-W27,115.0,NaN,663,0,0,44.48,0.25
8030,CP-005,Computer Peripherals,2024-W35,61.0,551.0,490,0,0,25.49,0.25
8031,MA-006,Mobile Accessories,2024-W29,133.0,1155.0,1022,0,0,19.49,0.25
8032,SH-009,Smart Home,2025-W12,8.0,325.0,317,0,0,28.70,0.25


## Data type

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8034 entries, 0 to 8033
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sku_id             8034 non-null   str    
 1   product_category   8034 non-null   str    
 2   week_id            8034 non-null   str    
 3   sales_qty          7759 non-null   float64
 4   inventory_begin    7751 non-null   float64
 5   inventory_end      8034 non-null   int64  
 6   receipts           8034 non-null   int64  
 7   lead_time_days     8034 non-null   int64  
 8   unit_cost          8034 non-null   float64
 9   holding_cost_rate  8034 non-null   float64
dtypes: float64(4), int64(3), str(3)
memory usage: 627.8 KB


## Statistical summary

In [4]:
df.describe()

,sales_qty,inventory_begin,inventory_end,receipts,lead_time_days,unit_cost,holding_cost_rate
count,7759.000000,7751.000000,8034.000000,8034.000000,8034.000000,8034.000000,8034.000000
mean,96.656915,1146.658496,1142.966517,93.372417,4.662061,45.121206,0.265985
std,125.048297,951.978088,951.384098,279.805016,10.959504,31.391823,0.020582
min,-351.000000,0.000000,0.000000,0.000000,0.000000,5.650000,0.250000
25%,31.000000,537.000000,536.250000,0.000000,0.000000,19.490000,0.250000
50%,61.000000,864.000000,861.000000,0.000000,0.000000,41.930000,0.250000
75%,126.000000,1370.000000,1358.000000,0.000000,0.000000,57.050000,0.280000
max,3530.000000,5145.000000,5145.000000,2303.000000,60.000000,131.410000,0.300000


## Time Range
- `week_id` covers 2023-W01 to 2025-W52 (3 years, 156 weeks) — matches Data Dictionary

In [5]:
digits = df['week_id'].str.findall(r'\d+')
week_clean = digits.apply(lambda x: sorted(x, key=len)[1] + '-W' + sorted(x, key=len)[0])
print(f"Earliest: {week_clean.min()}")
print(f"Latest:   {week_clean.max()}")

Earliest: 2023-W01
Latest:   2025-W52


# 2. Data Quality Check

## Null value check
- `Sales_qty`: 275 rows are null (3.42%)   
- `inventory_begin`: 283 rows are null (3.52%)

In [6]:
pd.DataFrame({
    'null_count': df.isna().sum(),
    'null_pct': (df.isna().sum() / df.shape[0] * 100).round(2)
})

,null_count,null_pct
sku_id,0,0.00
product_category,0,0.00
week_id,0,0.00
sales_qty,275,3.42
inventory_begin,283,3.52
inventory_end,0,0.00
receipts,0,0.00
lead_time_days,0,0.00
unit_cost,0,0.00
holding_cost_rate,0,0.00


## Duplicate Check
- 234 duplicate rows based on `sku_id` + `week_id` (before normalizing sku_id casing)

In [7]:
print(df.duplicated(subset=['sku_id', 'week_id']).sum())

234


## Numerical Anomaly check

`sales_qty`: 
- 80 rows of negative values, invalid — sales cannot be negative
- 20 rows > 600 — outliers, normal range is 0~600 

In [8]:
print(f"Negative: {(df['sales_qty'] < 0).sum()}")
print(f"Outliers: {(df['sales_qty'] > 600).sum()}")

Negative: 80
Outliers: 20


`inventory_begin`, `inventory_end`:

`receipts`:

`lead_time_days`

`unit_cost`

`holding_cost_rate`

## Inconsistency Check

### Unique value counts vs expected:
- `sku_id`: 241 unique values, expected 50 ⚠
- `product_category`: 35 unique values, expected 5 ⚠
- `week_id`: 277 unique values, expected 156 ⚠
- `unit_cost`: 50 unique values, expected 50 (one per SKU)
- `holding_cost_rate`: 3 unique values, expected 3 (0.25, 0.28, 0.30)
- `lead_time_days`: 53 unique values, expected range 14–56 (0 when no receipt)

In [9]:
df.nunique()

sku_id                241
product_category       35
week_id               277
sales_qty             530
inventory_begin      2276
inventory_end        2323
receipts              756
lead_time_days         53
unit_cost              50
holding_cost_rate       3
dtype: int64


### Inconsistency types found:
- `product_category`: casing (Audio/AUDIO/audio), abbreviations (Aud., Wear., Mob. Acc.), aliases (PC Peripherals, SmartHome), leading/trailing spaces
- `sku_id`: casing (aU-010, rnA-005), underscore vs dash (WR_007, SH_010), letter O vs zero (WR-O02, MA-0O5), leading spaces ( CP-002)
- `week_id`: reversed format (W09-2024 vs 2023-W28), slash vs dash (2024/W37 vs 2024-W37)

In [10]:
for col in ['product_category', 'sku_id', 'week_id']:
    print(f'\n--- {col} ---')
    display(df[col].unique())


--- product_category ---


<StringArray>
[           'Smart Home',             'Wearables',                 'Audio',
    'Mobile Accessories',                 'Wear.',  'Computer Peripherals',
             'WEARABLES',            'SMART HOME',    'mobile accessories',
  'computer peripherals',  'COMPUTER PERIPHERALS',                 'audio',
            ' Wearables',        'PC Peripherals',           'Smart Home ',
                 'AUDIO',    'MOBILE ACCESSORIES',      'Wearable Devices',
        'Audio Products',             'wearables',    'Smart Home Devices',
           'Mobile Acc.',   ' Mobile Accessories',             'SmartHome',
 'Computer Peripherals ',                  'Aud.',           ' Smart Home',
            'smart home',             'Mob. Acc.',         'Comp. Periph.',
 ' Computer Peripherals',                'Audio ',            'Wearables ',
                ' Audio',   'Mobile Accessories ']
Length: 35, dtype: str


--- sku_id ---


<StringArray>
[ 'SH-005',  'WR_007',  'aU-010',  'WR-005',  'MA-003',  'WR-010',  'WR-O02',
  'SH-002',  'WR-006',  'SH-007',
 ...
 'rnA-005',  'MA-0O5',  'WR_002',  'MA-0O3',  'SH_010', ' CP-002',  'AU_010',
  'WR_001',  'CP_006',  'MA-O05']
Length: 241, dtype: str


--- week_id ---


<StringArray>
['2023-W28', '2024-W12', '2024-W17', '2023-W52', '2023-W07', '2024-W20',
 '2024-W31', '2023-W45', '2024-W19', '2025-W45',
 ...
 'W22-2025', '2024/W37', '2023/W03', 'W39-2025', '2024/W35', 'W26-2025',
 'W09-2024', 'W35-2025', '2023/W11', '2023/W25']
Length: 277, dtype: str

# 3. Logic Consistency

- 225 rows have inventory balance inconsistency (begin - sales + receipts ≠ end), excluding null rows

In [11]:
df_no_null = df.dropna(subset=['sales_qty', 'inventory_begin'])
print((df_no_null['inventory_begin'] - df_no_null['sales_qty'] + df_no_null['receipts'] != df_no_null['inventory_end']).sum())

225


# Data standardization